# Follow-up: `complaints.affected_product_id` owner mismatch (100%)

The main `complaints` deep dive found that **every single complaint with a populated `affected_product_id` (44,570/44,570) points to a product owned by someone other than the complaint's own `customer_id`.** That's `checks.py`'s existing `product_owner_mismatch` check, reproduced directly and confirmed.

Before concluding anything, this notebook checks four things that would distinguish "real synthetic-generation bug" from "actually explainable":

1. Does the **same check** on `transactions`/`digital_events` (the other two tables `relationship_checks()` runs this on) also show a high mismatch rate, or is `complaints` uniquely broken?
2. When a complaint's own `customer_id` **does** have products of their own, does `affected_product_id` ever land on one of THEIR OWN products, or is it always a stranger's -- even when a correct match was available?
3. Is the mismatch **temporally implausible** too (e.g. does the "wrong" product even exist yet as of the complaint's `creation_date`), which would rule out any story where the product ID just refers to something opened later?
4. Does the mismatch look **structured** (e.g. same offset every time, clustered on specific customers) or **uniformly random** across the whole `products` table -- structure would suggest a join/shuffle bug with a specific mechanism; uniform randomness points more toward `affected_product_id` being generated independently of `customer_id` altogether.

Run top to bottom.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    return con.execute(sql, params or []).df()

## 1. Same `product_owner_mismatch` check reproduced for `transactions` and `digital_events` -- is this complaints-specific?

In [ ]:
for table, product_field in [("transactions", "product_id"), ("digital_events", "product_id")]:
    result = q(f"""
        SELECT
            count(*) AS total_linked,
            count(*) FILTER (WHERE c.customer_id = p.customer_id) AS owner_matches,
            count(*) FILTER (WHERE c.customer_id <> p.customer_id) AS owner_mismatches
        FROM bronze.{table} c
        JOIN bronze.products p ON p.product_id = c.{product_field}
        WHERE c.{product_field} IS NOT NULL AND c.customer_id IS NOT NULL
    """)
    result.insert(0, "table", table)
    print(f"{table}:")
    display(result)

## 2. When a complaint's `customer_id` owns products themselves, does `affected_product_id` ever land on one of their own?

If `affected_product_id` were even loosely tied to the real customer (e.g. picking the wrong one of several), we'd expect at least some hits when the customer has multiple products. If it's fully independent of `customer_id`, we'd expect ~0% even here.

In [ ]:
q("""
    WITH customers_with_products AS (
        SELECT DISTINCT customer_id FROM bronze.products
    )
    SELECT
        count(*) AS complaints_where_customer_owns_products,
        count(*) FILTER (WHERE c.affected_product_id IN (
            SELECT product_id FROM bronze.products p2 WHERE p2.customer_id = c.customer_id
        )) AS affected_product_is_actually_theirs
    FROM bronze.complaints c
    JOIN customers_with_products cwp ON cwp.customer_id = c.customer_id
    WHERE c.affected_product_id IS NOT NULL
""")

## 3. Temporal plausibility -- does the (wrong-owner) `affected_product_id` even exist as of the complaint's `creation_date`?

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE p.opening_date IS NOT NULL AND TRY_CAST(p.opening_date AS TIMESTAMP) > TRY_CAST(c.creation_date AS TIMESTAMP)) AS product_opened_after_complaint,
        count(*) FILTER (WHERE p.opening_date IS NOT NULL AND TRY_CAST(p.opening_date AS TIMESTAMP) <= TRY_CAST(c.creation_date AS TIMESTAMP)) AS product_existed_by_complaint
    FROM bronze.complaints c
    JOIN bronze.products p ON p.product_id = c.affected_product_id
    WHERE c.affected_product_id IS NOT NULL
      AND c.customer_id <> p.customer_id
""")

## 4. Is the mismatch structured or uniformly random?

Checking two things: (a) whether the SAME `affected_product_id` repeats suspiciously across many complaints (would suggest a fixed/default value rather than a real per-row assignment), and (b) whether the distribution of mismatched owners' `product_type`/`currency` looks like the full `products` population (uniform/random) or is skewed toward something specific.

In [ ]:
print("Most repeated affected_product_id values (top 10):")
display(q("""
    SELECT affected_product_id, count(*) AS n
    FROM bronze.complaints WHERE affected_product_id IS NOT NULL
    GROUP BY 1 ORDER BY n DESC LIMIT 10
"""))
print()
print("Total distinct affected_product_id values used vs. total complaints with one populated:")
display(q("""
    SELECT count(*) AS complaints_with_product, count(DISTINCT affected_product_id) AS distinct_products_used
    FROM bronze.complaints WHERE affected_product_id IS NOT NULL
"""))

In [ ]:
print("product_type distribution among affected products (complaints):")
display(q("""
    SELECT p.product_type, count(*) AS n
    FROM bronze.complaints c JOIN bronze.products p ON p.product_id = c.affected_product_id
    WHERE c.affected_product_id IS NOT NULL
    GROUP BY 1 ORDER BY n DESC
"""))
print()
print("product_type distribution across ALL of bronze.products, for comparison:")
display(q("SELECT product_type, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC"))

## 5. Sample -- 15 mismatched rows, side by side

Complaint's own `customer_id`, the `affected_product_id`'s real owner, and both parties' basic info, to eyeball whether anything looks structurally related (e.g. same segment, same branch, sequential IDs) rather than fully random.

In [ ]:
q("""
    SELECT
        c.complaint_id, c.customer_id AS complainant_customer_id, c.affected_product_id,
        p.customer_id AS actual_product_owner, p.product_type, p.currency,
        cust1.segment AS complainant_segment, cust2.segment AS owner_segment,
        cust1.registration_branch_id AS complainant_branch, cust2.registration_branch_id AS owner_branch
    FROM bronze.complaints c
    JOIN bronze.products p ON p.product_id = c.affected_product_id
    JOIN bronze.customers cust1 ON cust1.customer_id = c.customer_id
    JOIN bronze.customers cust2 ON cust2.customer_id = p.customer_id
    WHERE c.affected_product_id IS NOT NULL
    ORDER BY c.complaint_id
    LIMIT 15
""")

In [ ]:
con.close()
print("closed")